# Notebook 11 — Static visual evidence report

Purpose: build a local, self-contained HTML report and PNG figures from completed JEPA study artifacts, without training or model inference. Inputs: the frozen plan, evaluation summaries, saved development predictions/features, per-update training CSVs, and the pinned Food101 image adapter. Outputs: `artifacts/notebook_11/<run_id>/stages/learning/visual_report/`. Expected runtime: 1–3 minutes on CPU after all scores finish.

This notebook exports definitions only. Checkpoint location: existing study checkpoints under `checkpoints/<run_id>/`; this code never modifies them. Exact resume: rerun `build_visual_report(ROOT, artifact, plan, bundle, result)` after all registered evaluations are COMPLETE. Report writes use one writer and interruption-safe file replacement. If `.writer.lock` remains after interruption, verify its recorded process is gone before removing the stale lock. The latest report does not alter or replace training state.

In [ ]:
from pathlib import Path
import os, json, types
ROOT=Path(os.environ.get("FMM_JEPA_ROOT", "/home/benjamin/workspace/ahr_go/fmm_jeppa")).resolve()
_boot={"__file__":str(ROOT/"fmm_jepa/bootstrap.ipynb")}
for cell in json.loads((ROOT/"fmm_jepa/bootstrap.ipynb").read_text())["cells"]:
    if cell["cell_type"]=="code" and "ahr-export" in cell.get("metadata",{}).get("tags",[]):
        exec(compile("".join(cell["source"]),"bootstrap#ahr-export","exec"),_boot)
_boot["install_notebook_package"](ROOT)

__file__=str(ROOT/"experiments/11_manifold_long/visualization.ipynb")


## Validate and reconcile evidence

Use saved prediction counts to verify development accuracy, preserve101 classes and exact ordered image IDs, and reject changed feature-cache hashes.

In [ ]:
"""Static local evidence report: cached features only, no training or inference."""
from pathlib import Path
import base64, csv, hashlib, html, io, json, math, os, tempfile, time
import numpy as np
import torch
from PIL import Image
import matplotlib
os.environ["MPLCONFIGDIR"]=str(Path(__file__).resolve().parents[2]/".cache/matplotlib11")
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.stats import t as student_t

VARIANTS=("vit","spatial_kernel","learned_kernel")
DISPLAY_NAMES={"vit":"ViT","spatial_kernel":"Spatial kernel","learned_kernel":"Learned kernel"}
COLORS={"vit":"#2D6698","spatial_kernel":"#B8791B","learned_kernel":"#A94F83"}
CLASS_COLORS=("#2D6698","#B8791B","#A94F83","#6C7835","#DC7553","#25485B","#85603F","#704C7B","#728B80","#993D52")
CLASS_MARKERS=("o","s","^","D","v","P","X","<",">","*")
VISUAL_SEED=2026141101

def _json(path):
    return json.loads(Path(path).read_text())

def _sha(path):
    digest=hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda:stream.read(1024*1024),b""):digest.update(block)
    return digest.hexdigest()

def _canonical_hash(value):
    return hashlib.sha256(json.dumps(value,sort_keys=True,separators=(",",":"),allow_nan=False).encode()).hexdigest()

def _atomic_text(path,text):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    temporary=path.with_name(path.name+".partial")
    with temporary.open("w") as stream:
        stream.write(text);stream.flush();os.fsync(stream.fileno())
    os.replace(temporary,path)

def _write_json(path,value):
    _atomic_text(path,json.dumps(value,indent=2,sort_keys=True,allow_nan=False)+"\n")

def _deadline(deadline_unix):
    if deadline_unix is not None and time.time()>=float(deadline_unix):
        raise TimeoutError("Visual report deadline reached; no fabricated or incomplete PASS report")

def _record_path(root,record):
    path=Path(record["path"])
    if not path.is_absolute():path=Path(root)/path
    if not path.resolve().is_relative_to(Path(root).resolve()):raise ValueError("Evidence path escapes project")
    if _sha(path)!=record["sha256"]:raise ValueError("Saved evidence checksum differs: "+str(path))
    return path

def _source_artifact(artifact,plan):
    if plan.get("visualization_fixture_source"):
        if "Notebook 10" not in plan["visualization_fixture_source"]:raise ValueError("Unrecognized visual fixture")
        return Path(plan["visualization_source_artifact"])
    return Path(artifact)

def _prediction_rows(root,source,saved):
    candidate=source/"predictions"/(saved["model_id"]+".csv")
    if not candidate.exists():candidate=Path(saved["probe"]["summary"]["predictions_path"])
    if not candidate.resolve().is_relative_to(Path(root).resolve()):raise ValueError("Prediction path escapes project")
    with candidate.open() as stream:rows=list(csv.DictReader(stream))
    if not rows or len({row["sample_id"] for row in rows})!=len(rows):raise ValueError("Development predictions require unique IDs")
    labels=[int(row["label"]) for row in rows]
    if set(labels)!=set(range(101)):raise ValueError("All 101 Food101 development classes must remain represented")
    score=100*sum(int(row["correct_top1"]) for row in rows)/len(rows)
    measured=100*float(saved["probe"]["summary"]["metrics"]["top1"])
    if abs(score-measured)>1e-4:raise ValueError("Development accuracy does not reconcile with predictions")
    return rows,score,candidate

def evidence_rows(root,artifact,plan,result,require_complete=True):
    source=_source_artifact(artifact,plan)
    index={};rows=[];provenance=[]
    expected={(v,int(s),int(u)) for v in VARIANTS for s in plan["seeds"] for u in [0]+list(plan["milestones"])}
    for saved in result.get("evaluations",{}).values():
        key=(saved["variant"],int(saved["seed"]),int(saved["updates"]))
        if key in index or key not in expected:raise ValueError("Unexpected or duplicate evaluation identity")
        summary=saved["probe"]["summary"]
        if int(summary["completed_epochs"])!=20 or summary.get("selection")!="fixed_final_epoch":raise ValueError("Report requires fixed epoch 20 probes")
        if int(summary["budget"]["num_classes"])!=101:raise ValueError("Report requires101-class probes")
        predictions,score,path=_prediction_rows(root,source,saved)
        if [r["sample_id"] for r in predictions]!=list(result["membership"]["development_ids"]):raise ValueError("Prediction IDs differ from fixed development membership")
        cost=saved.get("training_cost") or {};perf=cost.get("performance") or {}
        peak=perf.get("peak_cuda_allocated_bytes")
        row={"model_id":saved["model_id"],"variant":key[0],"seed":key[1],"updates":key[2],"top1_percent":score,
             "development_examples":len(predictions),"classes":101,
             "median_step_seconds":(perf.get("complete_jepa_step_seconds") or {}).get("median"),
             "peak_GiB":None if peak is None else float(peak)/1024**3,
             "cumulative_step_seconds":cost.get("cumulative_step_seconds"),
             "encoder_parameters":saved["parameter_counts"]["context_active"],
             "trainable_parameters":saved["parameter_counts"]["trainable_active"]}
        index[key]=saved;rows.append(row)
        provenance.append({"model_id":row["model_id"],"prediction_csv":str(path),"prediction_sha256":_sha(path)})
    if require_complete and (result.get("status")!="COMPLETE" or set(index)!=expected):raise ValueError("All registered initial/milestone/final scores must finish before final report")
    lookup={(r["variant"],r["seed"],r["updates"]):r for r in rows}
    for row in rows:
        initial=lookup.get((row["variant"],row["seed"],0))
        row["gain_percentage_points"]=None if initial is None else row["top1_percent"]-initial["top1_percent"]
    return sorted(rows,key=lambda r:(r["variant"],r["seed"],r["updates"])),index,provenance

def paired_statistics(values,seeds):
    values=np.asarray(values,dtype=np.float64);n=len(values)
    if n<2 or len(seeds)!=n or len(set(seeds))!=n:raise ValueError("Distinct paired training seeds are required")
    mean=float(values.mean());se=float(values.std(ddof=1))/math.sqrt(n)
    half=float(student_t.ppf(.975,n-1))*se
    return {"n":n,"df":n-1,"mean_percentage_points":mean,"ci95_percentage_points":[mean-half,mean+half],
            "paired_seed_deltas":[{"seed":int(seed),"delta_percentage_points":float(value)} for seed,value in zip(seeds,values)],
            "method":"Unadjusted two-sided95% paired training-seed Student t interval, conditional on reused development images"}

def _plot_style():
    plt.rcParams.update({"font.family":"DejaVu Sans","font.size":11,"axes.titlesize":12,"axes.labelsize":11,
        "axes.spines.top":False,"axes.spines.right":False,"axes.grid":True,"grid.color":"#E2E5E8",
        "grid.linewidth":.65,"axes.axisbelow":True,"figure.facecolor":"white","axes.facecolor":"white","savefig.facecolor":"white"})

def _save_figure(figure,directory,name,title,note):
    directory=Path(directory);directory.mkdir(parents=True,exist_ok=True)
    path=directory/(name+".png");temporary=path.with_name(path.stem+".partial.png")
    figure.savefig(temporary,dpi=150,bbox_inches="tight");os.replace(temporary,path);plt.close(figure)
    return {"name":name,"path":str(path),"sha256":_sha(path),"title":title,"note":note}


## Render performance and uncertainty

Show raw loss and trailing means, every training seed, paired confidence intervals, own-initialization gains, and measured resource costs. Missing costs remain UNKNOWN.

In [ ]:
def training_loss_figure(root,artifact,plan,directory):
    _plot_style();source=_source_artifact(artifact,plan)
    figure,axes=plt.subplots(1,3,figsize=(14.4,4.5),sharex=True,sharey=True)
    sources=[];counts={}
    for column,variant in enumerate(VARIANTS):
        ax=axes[column];counts[variant]=0
        for number,seed in enumerate(plan["seeds"]):
            arm=f"s{seed}-{variant}";paths=sorted((source/"arms"/arm).glob("*_training_metrics.csv"))
            if not paths:continue
            if len(paths)!=1:raise ValueError("Ambiguous training metrics source")
            with paths[0].open() as stream:rows=list(csv.DictReader(stream))
            if not rows:continue
            x=np.asarray([int(row["successful_updates"]) for row in rows]);y=np.asarray([float(row["loss"]) for row in rows])
            if not np.isfinite(y).all() or len(set(x))!=len(x):raise ValueError("Invalid per-update training loss")
            ax.plot(x,y,color=COLORS[variant],alpha=.14,lw=.7)
            window=min(25,len(y));smoothed=np.convolve(y,np.ones(window)/window,mode="valid")
            ax.plot(x[window-1:],smoothed,color=COLORS[variant],alpha=.7,lw=1.1,linestyle=("-","--",":","-.")[number%4])
            counts[variant]+=len(rows);sources.append({"path":str(paths[0]),"sha256":_sha(paths[0]),"updates":len(rows)})
        ax.set(title=DISPLAY_NAMES[variant],xlabel="Successful optimizer updates")
        if not counts[variant]:ax.text(.5,.5,"Training metrics not yet saved",transform=ax.transAxes,ha="center")
    axes[0].set_ylabel("JEPA SmoothL1 loss")
    figure.suptitle("Training objective over the full run",y=1.02)
    note="Faint curves are raw per-update loss; darker curves are trailing 25-update means (or all available updates if fewer). Each encoder predicts its own evolving EMA teacher: loss is a within-run optimization diagnostic, not a common architectural quality scale."
    return _save_figure(figure,directory,"training_loss","JEPA training loss",note),sources

def metric_figures(rows,plan,directory):
    if not rows:return [],{}
    _plot_style();steps=[0]+list(plan["milestones"]);seeds=list(map(int,plan["seeds"]));lookup={(r["variant"],r["seed"],r["updates"]):r for r in rows}
    records=[];statistics={}
    figure,axes=plt.subplots(1,3,figsize=(14.4,4.5),sharey=True)
    ymax=max([r["top1_percent"] for r in rows]+[1.0])+1
    for ax,variant in zip(axes,VARIANTS):
        for number,seed in enumerate(seeds):
            present=[u for u in steps if (variant,seed,u) in lookup]
            ax.plot(present,[lookup[(variant,seed,u)]["top1_percent"] for u in present],color=COLORS[variant],alpha=.38,lw=1,
                    marker=("o","s","^","D","v","P")[number%6],markersize=5,label=str(seed))
        means=[np.mean([lookup[(variant,seed,u)]["top1_percent"] for seed in seeds if (variant,seed,u) in lookup]) if any((variant,s,u) in lookup for s in seeds) else np.nan for u in steps]
        ax.plot(steps,means,color=COLORS[variant],lw=2.7,marker="o",markersize=6,label="Seed mean")
        ax.axhline(100/101,color="#555555",lw=.8,linestyle=":")
        ax.set(title=DISPLAY_NAMES[variant],xlabel="Pretraining optimizer updates",ylim=(0,ymax),xticks=steps)
    axes[0].set_ylabel("Development top1 accuracy (%)")
    axes[-1].legend(title="Training seed",loc="upper left",bbox_to_anchor=(1.02,1),fontsize=8)
    figure.suptitle("Fixed epoch 20 frozen EMA probes: every training seed",y=1.02)
    records.append(_save_figure(figure,directory,"development_accuracy","Development accuracy by seed",
        "All 101 classes and the same fixed development IDs. Each point uses its own fixed epoch 20 linear probe with train-fitted feature standardization. Dotted reference is uniform101-class chance (0.990%). Seed means describe these runs; heldout/test evaluation remains NOT_RUN."))
    pairs=(("spatial_kernel","vit"),("learned_kernel","vit"),("learned_kernel","spatial_kernel"))
    compare_steps=[u for u in steps if u>0 and all((v,s,u) in lookup for v in VARIANTS for s in seeds)]
    if compare_steps:
        figure,axes=plt.subplots(1,len(compare_steps),figsize=(7.3*len(compare_steps),4.5),squeeze=False)
        for ax,step in zip(axes.flat,compare_steps):
            labels=[]
            for number,(candidate,control) in enumerate(pairs):
                values=[lookup[(candidate,s,step)]["top1_percent"]-lookup[(control,s,step)]["top1_percent"] for s in seeds]
                stats=paired_statistics(values,seeds);name=f"{candidate}_minus_{control}_u{step}";statistics[name]=stats
                low,high=stats["ci95_percentage_points"];mean=stats["mean_percentage_points"]
                ax.errorbar(mean,number,xerr=[[mean-low],[high-mean]],color=COLORS[candidate],fmt="D",ms=7,lw=2,capsize=5)
                offsets=np.linspace(-.13,.13,len(values))
                ax.scatter(values,number+offsets,color=COLORS[candidate],s=27,facecolors="none",zorder=3)
                labels.append(DISPLAY_NAMES[candidate]+" minus\n"+DISPLAY_NAMES[control])
            ax.axvline(0,color="#555555",lw=1);ax.set(yticks=range(len(pairs)),yticklabels=labels,xlabel="Paired accuracy difference (percentage points)",title=f"After {step:,} updates")
            ax.invert_yaxis()
        figure.suptitle("Architecture comparisons with paired training-seed uncertainty",y=1.02)
        figure.tight_layout()
        records.append(_save_figure(figure,directory,"paired_comparisons","Paired architecture comparisons",
            f"Diamonds show the paired mean; bars show unadjusted 95% Student t confidence intervals across {len(seeds)} training seeds (df={len(seeds)-1}); open points are the individual paired differences. These intervals condition on the shared development set and do not measure dataset-sampling uncertainty. Multiple comparisons are exploratory."))
    gain_steps=[u for u in steps if u>0 and all(lookup.get((v,s,u),{}).get("gain_percentage_points") is not None for v in VARIANTS for s in seeds)]
    if gain_steps:
        figure,axes=plt.subplots(1,len(gain_steps),figsize=(7.2*len(gain_steps),4.5),squeeze=False,sharey=True)
        for ax,step in zip(axes.flat,gain_steps):
            for number,variant in enumerate(VARIANTS):
                values=[lookup[(variant,s,step)]["gain_percentage_points"] for s in seeds];stats=paired_statistics(values,seeds)
                statistics[f"{variant}_learning_u{step}"]=stats;low,high=stats["ci95_percentage_points"];mean=stats["mean_percentage_points"]
                ax.errorbar(number,mean,yerr=[[mean-low],[high-mean]],color=COLORS[variant],fmt="D",ms=7,lw=2,capsize=5)
                ax.scatter(number+np.linspace(-.14,.14,len(values)),values,color=COLORS[variant],s=30,facecolors="none")
            ax.axhline(0,color="#555555",lw=1);ax.set(xticks=range(3),xticklabels=[DISPLAY_NAMES[v] for v in VARIANTS],title=f"After {step:,} updates")
        axes[0,0].set_ylabel("Gain from own initialization (percentage points)")
        figure.suptitle("Learning gains relative to each encoder's initialization",y=1.02)
        records.append(_save_figure(figure,directory,"learning_gains","Learning gains from initialization",
            "Trained probe accuracy minus the matching untrained encoder probe, paired within training seed. Diamonds and 95% t intervals summarize the seed differences; open points retain every seed. Feature finiteness or a falling JEPA loss alone does not establish learned task improvement."))
    final=max(steps);cost_rows=[r for r in rows if r["updates"]==final]
    if cost_rows:
        figure,axes=plt.subplots(1,3,figsize=(14.4,4.5))
        for ax,key,label in zip(axes,("median_step_seconds","peak_GiB","encoder_parameters"),("Complete JEPA step (seconds)","Peak allocated CUDA memory (GiB)","Active encoder parameters (millions)")):
            for number,variant in enumerate(VARIANTS):
                values=[float(r[key])/(1e6 if key=="encoder_parameters" else 1) for r in cost_rows if r["variant"]==variant and r[key] is not None]
                if values:
                    ax.bar(number,float(np.mean(values)),color=COLORS[variant],width=.55,alpha=.7)
                    ax.scatter(number+np.linspace(-.13,.13,len(values)),values,color="#333333",s=25,zorder=3)
                else:ax.text(number,.03,"UNKNOWN",rotation=90,ha="center",transform=ax.get_xaxis_transform(),fontsize=9)
            ax.set(xticks=range(3),xticklabels=["ViT","Spatial\nkernel","Learned\nkernel"],ylabel=label,ylim=(0,None))
        figure.suptitle("Measured training costs at the final milestone",y=1.02)
        records.append(_save_figure(figure,directory,"training_cost","Training compute, memory and capacity",
            "Bars are seed means; points are each training run. Step medians include data preprocessing plus full JEPA computation and exclude checkpoint/diagnostic I/O. Memory is measured peak allocated CUDA memory, excludes reserved cache/context, and follows the saved measurement scope. Missing measurements stay UNKNOWN. Active capacities differ."))
    return records,statistics


## Inspect images and representation geometry

The first registered seed and final milestone are fixed independently of quality. Use the same deterministic balanced development subset for each encoder, separate centered PCA projections, and the same queries for full-development cosine neighbors. Read no heldout images.

In [ ]:
def _load_development_cache(root,saved,expected_ids,dataset_fingerprint):
    record=saved["feature_caches"]["development"];path=_record_path(root,record)
    cache=torch.load(path,map_location="cpu",weights_only=True)
    features=cache["features"].detach().float().cpu().numpy();labels=cache["labels"].detach().cpu().numpy().astype(int)
    ids=list(cache["sample_ids"]);identity=cache["identity"]
    if ids!=list(expected_ids) or list(identity["sample_ids"])!=ids:raise ValueError("Cached development identities differ")
    if identity["dataset_revision"]!=dataset_fingerprint or identity["pooling"]!="mean_final_layernorm_patches":raise ValueError("Cached feature source or pooling differs")
    if features.ndim!=2 or features.shape[0]!=len(ids) or len(labels)!=len(ids) or not np.isfinite(features).all():raise ValueError("Invalid cached development features")
    if set(labels.tolist())!=set(range(101)):raise ValueError("All 101 development classes required")
    return features,labels,ids,{"path":str(path),"sha256":record["sha256"],"identity":identity}

def _image_thumbnail(dataset,identifier,label):
    item=dataset.get(identifier)
    if int(item["label"])!=int(label):raise ValueError("Image source label disagrees with cached feature label")
    return np.asarray(item["image"].convert("RGB").resize((128,128),Image.Resampling.BILINEAR))

def _pca(features):
    centered=np.asarray(features,dtype=np.float64)-np.asarray(features,dtype=np.float64).mean(0)
    left,singular,right=np.linalg.svd(centered,full_matrices=False)
    coordinates=left[:,:2]*singular[:2]
    for dimension in range(2):
        pivot=int(np.argmax(np.abs(right[dimension])))
        if right[dimension,pivot]<0:coordinates[:,dimension]*=-1
    variance=singular**2
    explained=variance[:2]/variance.sum() if variance.sum()>0 else np.zeros(2)
    return coordinates,explained

def representation_figures(root,artifact,plan,bundle,result,index,directory,deadline_unix=None):
    _plot_style();seed=int(plan["seeds"][0]);step=max(map(int,plan["milestones"]));expected=result["membership"]["development_ids"]
    dataset=bundle["dataset"]
    if dataset.fingerprint!=result["dataset_fingerprint"] or len(dataset.class_names)!=101:raise ValueError("Visual image source differs from evaluation")
    caches={};sources=[];labels=ids=None
    for variant in VARIANTS:
        _deadline(deadline_unix)
        features,other_labels,other_ids,provenance=_load_development_cache(root,index[(variant,seed,step)],expected,result["dataset_fingerprint"])
        if labels is not None and (ids!=other_ids or not np.array_equal(labels,other_labels)):raise ValueError("Encoder image identities or labels differ")
        features=np.asarray(features,dtype=np.float64);norms=np.linalg.norm(features,axis=1,keepdims=True)
        if np.any(norms<=1e-12):raise ValueError("Cosine neighbors undefined for zero vectors")
        caches[variant]={"features":features,"normalized":features/norms};labels=other_labels;ids=other_ids
        sources.append({"variant":variant,"seed":seed,"updates":step,**provenance})
    rng=np.random.default_rng(VISUAL_SEED);selected_classes=sorted(rng.choice(np.arange(101),size=10,replace=False).tolist())
    positions=[]
    for label in selected_classes:
        candidates=np.flatnonzero(labels==label);chosen=rng.choice(candidates,size=min(20,len(candidates)),replace=False)
        positions.extend(sorted(chosen.tolist()))
    if len({sum(labels[positions]==label) for label in selected_classes})!=1:raise ValueError("PCA subset must be balanced across selected classes")
    query_positions=[next(p for p in positions if labels[p]==label) for label in selected_classes[:3]]
    thumbnail_positions=[next(p for p in positions if labels[p]==label) for label in selected_classes]
    image_cache={}
    def thumb(position):
        if position not in image_cache:image_cache[position]=_image_thumbnail(dataset,ids[position],labels[position])
        return image_cache[position]
    records=[]
    figure,axes=plt.subplots(2,5,figsize=(13,6.3))
    for ax,position in zip(axes.flat,thumbnail_positions):
        ax.imshow(thumb(position));ax.axis("off");ax.set_title(dataset.class_names[int(labels[position])].replace("_"," "),fontsize=11)
    figure.suptitle("Fixed development-image examples from 10 seeded classes",y=1.01)
    records.append(_save_figure(figure,directory,"development_gallery","Actual development image examples",
        "Original development images resized to128×128 using the deterministic evaluation resize; no training augmentation. Classes and IDs selected with fixed seed 2026141101, independent of probe outcomes. Source labels describe Food101 metadata."))
    figure,axes=plt.subplots(1,3,figsize=(14.4,5.4))
    projections={}
    for ax,variant in zip(axes,VARIANTS):
        coordinates,explained=_pca(caches[variant]["features"][positions]);projections[variant]={"explained_variance_ratio":explained.tolist(),"coordinates":coordinates.tolist()}
        for number,label in enumerate(selected_classes):
            active=labels[positions]==label
            ax.scatter(coordinates[active,0],coordinates[active,1],color=CLASS_COLORS[number],marker=CLASS_MARKERS[number],s=24,alpha=.78,label=dataset.class_names[label].replace("_"," "))
        ax.set(title=DISPLAY_NAMES[variant],xlabel=f"PC1 ({100*explained[0]:.1f}% variance)",ylabel=f"PC2 ({100*explained[1]:.1f}% variance)")
    axes[-1].legend(loc="upper left",bbox_to_anchor=(1.02,1),fontsize=9,title="Food101 class")
    figure.suptitle(f"Same {len(positions)} balanced development images; seed {seed}, update {step:,}",y=1.02)
    records.append(_save_figure(figure,directory,"representation_pca","Two-dimensional representation projections",
        "Each panel independently fits centered PCA to the same balanced 10-class subset of raw pooled EMA features. Axes, scales and orientations belong to each encoder and are not aligned across panels. PCA preserves selected variance, not all distances; visual separation is descriptive and is not a classification score. Seed and final checkpoint were fixed by protocol."))
    neighbor_evidence=[]
    for query_number,query in enumerate(query_positions):
        _deadline(deadline_unix)
        figure,axes=plt.subplots(3,6,figsize=(13.2,7.4))
        for row,variant in enumerate(VARIANTS):
            normalized=caches[variant]["normalized"];similarities=normalized@normalized[query];similarities[query]=-np.inf
            nearest=np.argsort(-similarities,kind="stable")[:5]
            row_positions=[query]+nearest.tolist()
            for column,position in enumerate(row_positions):
                ax=axes[row,column];ax.imshow(thumb(position));ax.set_xticks([]);ax.set_yticks([]);ax.grid(False)
                for spine in ax.spines.values():spine.set_visible(False)
                label=dataset.class_names[int(labels[position])].replace("_"," ")
                ax.set_title(("Query: " if column==0 else f"#{column} cosine {similarities[position]:.4f}\n")+label,fontsize=9)
                if column==0:ax.set_ylabel(DISPLAY_NAMES[variant],fontsize=10,labelpad=8)
            neighbor_evidence.append({"query_id":ids[query],"query_label":int(labels[query]),"variant":variant,"seed":seed,"updates":step,
                "neighbors":[{"sample_id":ids[p],"label":int(labels[p]),"cosine":float(similarities[p])} for p in nearest]})
        figure.suptitle("Nearest development images for "+dataset.class_names[int(labels[query])].replace("_"," "),y=1.02)
        records.append(_save_figure(figure,directory,f"nearest_neighbors_{query_number+1}","Representation nearest neighbors: "+dataset.class_names[int(labels[query])].replace("_"," "),
            "Rows use the same query and the three final EMA encoders from the first registered training seed. Five neighbors are ranked by cosine similarity in raw mean-pooled feature space across all fixed development images; the query is excluded. No training/probe standardization or class labels enter neighbor ranking. Example choices are fixed, illustrative, and never used for model selection."))
    selected={"selection_seed":VISUAL_SEED,"training_seed":seed,"updates":step,"class_labels":selected_classes,
        "class_names":[dataset.class_names[label] for label in selected_classes],"sample_ids":[ids[p] for p in positions],
        "query_ids":[ids[p] for p in query_positions],
        "displayed_image_sources":[{"sample_id":ids[p],"label":int(labels[p]),"path":str(dataset.path_for(ids[p])),"encoded_sha256":_sha(dataset.path_for(ids[p]))} for p in sorted(image_cache)],"balanced_per_class":int(len(positions)/10),"pca":projections,"neighbors":neighbor_evidence}
    return records,selected,sources


## Build the local report or render saved progress

`build_visual_report` requires every registered score. `render_review` permits partial saved evidence and labels it IN_PROGRESS. HTML contains embedded PNGs and direct PNG downloads; there are no remote scripts or publication calls.

In [ ]:
def load_review_bundle(root,artifact,result):
    """Reopen pinned source metadata read-only; never prepare/download or read test pixels."""
    from fmm_jepa.food101_data import Food101Adapter
    root,artifact=Path(root),Path(artifact)
    manifest=_json(root/"data/food101/dataset_manifest.json")
    dataset=Food101Adapter(root/"data/food101",manifest)
    if dataset.fingerprint!=result["dataset_fingerprint"]:raise ValueError("Review dataset fingerprint differs")
    return {"dataset":dataset,"dev_ids":list(result["membership"]["development_ids"])}

def load_review_inputs(root,artifact):
    """Read completed or in-progress evidence without launching any worker."""
    root,artifact=Path(root),Path(artifact)
    if not (artifact/"plan.json").exists():return {"status":"NOT_STARTED","plan":None,"result":None}
    plan=_json(artifact/"plan.json")
    if (artifact/"stage_evaluation.json").exists():
        result=_json(artifact/"stage_evaluation.json")
        return {"status":result.get("status","UNKNOWN"),"plan":plan,"result":result}
    evaluations={};manifest_path=artifact/"stage_evaluation_manifest.json"
    if manifest_path.exists():
        manifest=_json(manifest_path)
        for model_id,record in manifest.get("completed",{}).items():
            path=_record_path(root,record);evaluations[model_id]=_json(path)
    split=_json(artifact/"split_manifest.json") if (artifact/"split_manifest.json").exists() else {}
    data=_json(artifact/"stage_data_manifest.json") if (artifact/"stage_data_manifest.json").exists() else {}
    result={"status":"IN_PROGRESS","run_id":plan["run_id"],"evaluations":evaluations,
        "membership":{"development_ids":split.get("selected_ids",{}).get("development",[])},
        "dataset_fingerprint":data.get("dataset_fingerprint")}
    return {"status":"IN_PROGRESS","plan":plan,"result":result}

def render_review(root,artifact,plan,result,bundle=None,directory=None,deadline_unix=None):
    """Render local figure artifacts directly from saved evidence; training remains read-only."""
    root,artifact=Path(root).resolve(),Path(artifact).resolve()
    directory=Path(directory) if directory else artifact/"review_figures"
    if not directory.resolve().is_relative_to(root):raise ValueError("Review outputs must remain under project")
    _deadline(deadline_unix)
    complete=result.get("status")=="COMPLETE"
    rows,index,provenance=evidence_rows(root,artifact,plan,result,require_complete=complete)
    loss,loss_sources=training_loss_figure(root,artifact,plan,directory)
    figures,statistics=metric_figures(rows,plan,directory);figures=[loss]+figures
    selected=None;feature_sources=[]
    if complete:
        if bundle is None:bundle=load_review_bundle(root,_source_artifact(artifact,plan),result)
        representation,selected,feature_sources=representation_figures(root,artifact,plan,bundle,result,index,directory,deadline_unix)
        figures.extend(representation)
    return {"status":"PASS" if complete else "IN_PROGRESS","figures":figures,"rows":rows,"paired_statistics":statistics,
        "visual_selection":selected,"prediction_sources":provenance,"training_sources":loss_sources,"feature_sources":feature_sources}

def _table(rows):
    columns=("variant","seed","updates","top1_percent","gain_percentage_points","median_step_seconds","peak_GiB")
    titles=("Encoder","Seed","Updates","Dev top1 (%)","Gain (pp)","Step (s)","Peak (GiB)")
    body=[]
    for row in rows:
        values=[]
        for name in columns:
            value=row.get(name)
            if name=="variant":value=DISPLAY_NAMES[value]
            elif value is None:value="UNKNOWN"
            elif isinstance(value,float):value=f"{value:.3f}"
            values.append("<td>"+html.escape(str(value))+"</td>")
        body.append("<tr>"+"".join(values)+"</tr>")
    return "<div class='table-scroll'><table><thead><tr>"+"".join("<th>"+v+"</th>" for v in titles)+"</tr></thead><tbody>"+"".join(body)+"</tbody></table></div>"

def build_visual_report(root,artifact,plan,bundle,result,deadline_unix=None):
    """Complete local self-contained HTML/PNG report after every registered probe finishes."""
    root,artifact=Path(root).resolve(),Path(artifact).resolve()
    if not artifact.is_relative_to(root/"artifacts/notebook_11"):raise ValueError("Notebook11 report outputs must stay in its own artifact tree")
    expected=len(plan["seeds"])*3*(1+len(plan["milestones"]))
    if result.get("status")!="COMPLETE" or len(result.get("evaluations",{}))!=expected:raise ValueError("All registered scores must complete before the visual report")
    folder=artifact/"visual_report";folder.mkdir(parents=True,exist_ok=True)
    lock=folder/".writer.lock"
    try:descriptor=os.open(lock,os.O_CREAT|os.O_EXCL|os.O_WRONLY)
    except FileExistsError:raise RuntimeError("A visual-report writer is already active; inspect the lock before resuming")
    try:
        with os.fdopen(descriptor,"w") as stream:stream.write(str(os.getpid()))
        rendered=render_review(root,artifact,plan,result,bundle,folder,deadline_unix)
        _deadline(deadline_unix)
        fixture=plan.get("visualization_fixture_source")
        title="Notebook 11 — Overnight JEPA representation study" if not fixture else "Notebook 10 fixture — visual report software validation"
        warning=("<p class='fixture'>This is completed Notebook 10 evidence used only to validate Notebook 11 report software. It contains no Notebook 11 experiment results.</p>" if fixture else "")
        scope=f"{len(plan['seeds'])} paired training seeds × 3 encoders; checkpoints {', '.join(str(u) for u in [0]+list(plan['milestones']))}; all 101 Food101 classes; {len(result['membership']['development_ids']):,} fixed development images. Each frozen EMA probe completes exactly 20 epochs."
        sections=[]
        for figure in rendered["figures"]:
            content=base64.b64encode(Path(figure["path"]).read_bytes()).decode()
            sections.append("<section><h2>"+html.escape(figure["title"])+"</h2><div class='figure-scroll'><img alt='"+html.escape(figure["title"],quote=True)+"' src='data:image/png;base64,"+content+"'></div><p>"+html.escape(figure["note"])+"</p><a download='"+Path(figure["path"]).name+"' href='data:image/png;base64,"+content+"'>Download PNG</a></section>")
        evidence={"schema":"notebook11-visual-report/v1","status":"PASS","source_run_id":result["run_id"],
            "fixture_source":fixture,"plan_sha256":_canonical_hash(plan),"result_sha256":_canonical_hash(result),
            "scope":scope,"official_test":"NOT_RUN","visualization_code_sha256":_sha(root/"experiments/11_manifold_long/visualization.ipynb"),
            "rows":rendered["rows"],"paired_statistics":rendered["paired_statistics"],"visual_selection":rendered["visual_selection"],
            "prediction_sources":rendered["prediction_sources"],"feature_sources":rendered["feature_sources"],"training_sources":rendered["training_sources"],
            "figures":rendered["figures"],"generated_unix":time.time()}
        _write_json(folder/"figure_evidence.json",evidence)
        document="<!doctype html><html lang='en'><meta charset='utf-8'><meta name='viewport' content='width=device-width, initial-scale=1'><title>"+html.escape(title)+"</title><style>body{margin:0;color:#25303a;background:#f6f7f9;font:16px/1.55 system-ui,sans-serif}main{max-width:1450px;margin:auto;padding:32px 24px}h1{font-size:30px;line-height:1.2}h2{font-size:22px}section{background:white;padding:24px;margin:24px 0}.figure-scroll{overflow-x:auto}img{display:block;width:100%;min-width:1100px;height:auto}p{max-width:1100px}a{color:#235f95}table{border-collapse:collapse;font-size:14px;width:100%}td,th{text-align:left;padding:9px;border-bottom:1px solid #dde2e8;white-space:nowrap}.table-scroll{overflow:auto}.fixture{padding:18px;background:#ffefc7;border-left:5px solid #b8791b}code{overflow-wrap:anywhere}@media(max-width:600px){main{padding:16px 8px}section{padding:14px 8px}h1{font-size:24px}}</style><main><h1>"+html.escape(title)+"</h1>"+warning+"<p>"+html.escape(scope)+"</p><p>Development results are exploratory. Training seeds quantify optimization variability on the reused development set; they do not establish performance on new test images. Official test/lockbox data remain untouched.</p>"+"".join(sections)+"<section><h2>Exact values and provenance</h2>"+_table(rendered["rows"])+"<p>Full chart data, paired intervals, feature-cache identities, image IDs and SHA-256 hashes are saved beside this report in <code>figure_evidence.json</code>. This HTML embeds every image and PNG download and needs no network connection. On small screens, scroll horizontally inside figures or download the full PNG.</p><p>Source run: <code>"+html.escape(result["run_id"])+"</code>; source result hash: <code>"+evidence["result_sha256"]+"</code>.</p></section></main></html>"
        path=folder/"report.html";_atomic_text(path,document)
        returned={"status":"PASS","html":str(path),"figures":[f["path"] for f in rendered["figures"]],
            "review_notebook":str(root/"notebooks/11_Overnight_Results_Review.ipynb"),"evidence":str(folder/"figure_evidence.json"),"fixture_source":fixture}
        _write_json(folder/"report_manifest.json",{**returned,"html_sha256":_sha(path),"figure_count":len(rendered["figures"]),"complete_scores":expected})
        return returned
    finally:
        lock.unlink(missing_ok=True)


In [ ]:
print("Ready: build_visual_report(root, artifact, plan, bundle, result, deadline_unix=None)")